# 🚤 Improved Boat Detection for Boca Raton Inlet

## Problem Diagnosis

Your original model had **very poor performance** (mAP@0.5 = 0.102). Here's why:

### 1. **SAHI Was Not Actually Slicing!**
Your images are 640×640 pixels, and you used:
```python
slice_height=640,
slice_width=640,
```
This means SAHI created **only 1 slice** (the whole image)! SAHI did nothing.

### 2. **Small Object Problem**
Looking at your metrics:
- **Small objects AP: 0.048** (terrible!)
- **Medium objects AP: 0.481** (okay)
- **Large objects AP: 0.700** (good)

Most boats in your inlet images appear **small** because the camera is far away.

### 3. **High Confidence Threshold**
You used `conf=0.3`. Distant/small boats often have lower confidence scores.

### 4. **Detection Gap**
- Ground truth boats: **169**
- Your predictions: **35** 
- **15 images had boats but ZERO predictions!**

---

## The Solution

| Parameter | Original | Fixed | Why |
|-----------|----------|-------|-----|
| Slice size | 640×640 | 256×256 | Creates ~9 slices, zooms into boat regions |
| Overlap | 0.2 | 0.3 | Better coverage at slice boundaries |
| Confidence | 0.3 | 0.15 | Catches more distant boats |
| Post-process | default | NMS + IOS | Better duplicate removal |

In [ ]:
# Install required packages
!pip install ultralytics sahi pycocotools -q

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict

from ultralytics import YOLO
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

## Configuration

⚠️ **Update these paths to match your setup**

In [ ]:
# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS
# =============================================================================
MODEL_PATH = "yolo11x.pt"
IMAGE_FOLDER = "/content/Boca_Raton"
GT_JSON = "/content/Boca_Raton/_annotations.coco.json"
OUTPUT_FOLDER = "/content/Boca_Raton/output_improved"
DEVICE = "cuda:0"  # Use GPU

# =============================================================================
# OPTIMIZED PARAMETERS - KEY CHANGES FROM YOUR ORIGINAL!
# =============================================================================
SAHI_SLICE_SIZE = 256    # ← Was 640 (no slicing!), now creates ~9 slices
SAHI_OVERLAP = 0.3       # ← Was 0.2, better boundary coverage
CONF_THRESHOLD = 0.15    # ← Was 0.3, catches more small/distant boats

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print("Configuration loaded!")
print(f"  Slice size: {SAHI_SLICE_SIZE}×{SAHI_SLICE_SIZE}")
print(f"  Overlap: {SAHI_OVERLAP}")
print(f"  Confidence threshold: {CONF_THRESHOLD}")

## Load and Analyze Ground Truth

In [ ]:
# Load ground truth
with open(GT_JSON) as f:
    gt = json.load(f)

print(f"Total images: {len(gt['images'])}")
print(f"Total annotations: {len(gt['annotations'])}")

# Analyze boat sizes - this shows WHY small object detection matters
boat_annotations = [ann for ann in gt['annotations'] if ann['category_id'] == 1]
boat_areas = [ann['bbox'][2] * ann['bbox'][3] for ann in boat_annotations]
boat_widths = [ann['bbox'][2] for ann in boat_annotations]
boat_heights = [ann['bbox'][3] for ann in boat_annotations]

print(f"\n📊 Boat Size Analysis:")
print(f"  Total boats: {len(boat_areas)}")
print(f"  \n  Bounding box areas (pixels²):")
print(f"    Min: {min(boat_areas):.0f}, Max: {max(boat_areas):.0f}, Mean: {np.mean(boat_areas):.0f}")
print(f"  \n  Widths (pixels):")
print(f"    Min: {min(boat_widths):.0f}, Max: {max(boat_widths):.0f}, Mean: {np.mean(boat_widths):.0f}")

# COCO size categories
small = len([a for a in boat_areas if a < 32*32])    # < 1024 px²
medium = len([a for a in boat_areas if 32*32 <= a < 96*96])  # 1024-9216 px²
large = len([a for a in boat_areas if a >= 96*96])   # >= 9216 px²

print(f"\n  COCO Size Categories:")
print(f"    🔴 Small (<32×32):  {small} boats ({small/len(boat_areas)*100:.1f}%)")
print(f"    🟡 Medium (32-96):  {medium} boats ({medium/len(boat_areas)*100:.1f}%)")
print(f"    🟢 Large (>96×96):  {large} boats ({large/len(boat_areas)*100:.1f}%)")

if small > medium + large:
    print(f"\n⚠️  MOST BOATS ARE SMALL! Standard YOLO will struggle.")
    print(f"    → This is why SAHI with small slices is critical.")

## Method 1: Standard YOLO (Baseline)

This is similar to your original approach, to establish a baseline.

In [ ]:
print("="*60)
print("METHOD 1: Standard YOLO (Baseline)")
print("="*60)

model = YOLO(MODEL_PATH)

predictions_baseline = []

for img_data in tqdm(gt['images'], desc="Standard YOLO"):
    img_path = os.path.join(IMAGE_FOLDER, img_data['file_name'])
    if not os.path.exists(img_path):
        continue
    
    results = model(img_path, conf=0.3, device=DEVICE, verbose=False)  # Original conf=0.3
    
    for result in results:
        boxes = result.boxes
        for i in range(len(boxes)):
            class_id = int(boxes.cls[i])
            if class_id == 8:  # YOLO boat class
                xyxy = boxes.xyxy[i].cpu().numpy()
                conf = float(boxes.conf[i])
                predictions_baseline.append({
                    "image_id": img_data['id'],
                    "category_id": 1,
                    "bbox": [float(xyxy[0]), float(xyxy[1]),
                            float(xyxy[2] - xyxy[0]), float(xyxy[3] - xyxy[1])],
                    "score": conf
                })

print(f"\nBaseline predictions: {len(predictions_baseline)}")

# Evaluate
coco_gt = COCO(GT_JSON)
if predictions_baseline:
    coco_dt = coco_gt.loadRes(predictions_baseline)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.params.catIds = [1]
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

## Method 2: Optimized SAHI (The Fix!)

This is the key improvement:
- **256×256 slices** instead of 640×640
- **30% overlap** instead of 20%
- **Lower confidence** (0.15) to catch small boats

In [ ]:
print("="*60)
print("METHOD 2: Optimized SAHI Detection")
print("="*60)
print(f"Slice size: {SAHI_SLICE_SIZE}×{SAHI_SLICE_SIZE}")
print(f"Overlap: {SAHI_OVERLAP}")
print(f"Confidence: {CONF_THRESHOLD}")

# Calculate expected slices for a 640x640 image
stride = SAHI_SLICE_SIZE * (1 - SAHI_OVERLAP)
expected_slices = ((640 - SAHI_SLICE_SIZE) / stride + 1) ** 2
print(f"\nExpected slices per image: ~{expected_slices:.0f}")
print(f"(Your original: 1 slice - no zooming!)")

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=CONF_THRESHOLD,
    device=DEVICE
)

predictions_sahi = []

for img_data in tqdm(gt['images'], desc="SAHI Detection"):
    img_path = os.path.join(IMAGE_FOLDER, img_data['file_name'])
    if not os.path.exists(img_path):
        continue
    
    result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=SAHI_SLICE_SIZE,
        slice_width=SAHI_SLICE_SIZE,
        overlap_height_ratio=SAHI_OVERLAP,
        overlap_width_ratio=SAHI_OVERLAP,
        postprocess_type="NMS",
        postprocess_match_metric="IOS",
        postprocess_match_threshold=0.5,
    )
    
    for obj in result.object_prediction_list:
        if obj.category.id == 8:  # YOLO boat class
            bbox = obj.bbox
            predictions_sahi.append({
                "image_id": img_data['id'],
                "category_id": 1,
                "bbox": [bbox.minx, bbox.miny, bbox.maxx - bbox.minx, bbox.maxy - bbox.miny],
                "score": obj.score.value
            })

print(f"\nSAHI predictions: {len(predictions_sahi)}")
print(f"Improvement: {len(predictions_sahi) - len(predictions_baseline):+d} detections")

# Evaluate
if predictions_sahi:
    coco_dt_sahi = coco_gt.loadRes(predictions_sahi)
    coco_eval_sahi = COCOeval(coco_gt, coco_dt_sahi, 'bbox')
    coco_eval_sahi.params.catIds = [1]
    coco_eval_sahi.evaluate()
    coco_eval_sahi.accumulate()
    coco_eval_sahi.summarize()

## Method 3: Multi-Scale Detection

Run detection at multiple image sizes and combine results.

In [ ]:
print("="*60)
print("METHOD 3: Multi-Scale Detection")
print("="*60)

def detect_multiscale(model, image_path, scales=[1.0, 1.5, 2.0], conf=0.2):
    """Detect at multiple scales and merge results."""
    img = cv2.imread(image_path)
    orig_h, orig_w = img.shape[:2]
    
    all_boxes = []
    all_scores = []
    
    for scale in scales:
        if scale == 1.0:
            results = model(image_path, conf=conf, verbose=False)
        else:
            # Upscale image
            scaled = cv2.resize(img, None, fx=scale, fy=scale, 
                              interpolation=cv2.INTER_LANCZOS4)
            temp_path = "/tmp/scaled_temp.jpg"
            cv2.imwrite(temp_path, scaled)
            results = model(temp_path, conf=conf, verbose=False)
        
        for result in results:
            boxes = result.boxes
            for i in range(len(boxes)):
                if int(boxes.cls[i]) == 8:  # boat
                    xyxy = boxes.xyxy[i].cpu().numpy()
                    if scale != 1.0:
                        xyxy = xyxy / scale  # Scale back
                    all_boxes.append(xyxy)
                    all_scores.append(float(boxes.conf[i]))
    
    # Apply NMS
    if not all_boxes:
        return []
    
    boxes_arr = np.array(all_boxes)
    scores_arr = np.array(all_scores)
    
    # Simple NMS
    keep = []
    order = scores_arr.argsort()[::-1]
    
    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break
        
        xx1 = np.maximum(boxes_arr[i, 0], boxes_arr[order[1:], 0])
        yy1 = np.maximum(boxes_arr[i, 1], boxes_arr[order[1:], 1])
        xx2 = np.minimum(boxes_arr[i, 2], boxes_arr[order[1:], 2])
        yy2 = np.minimum(boxes_arr[i, 3], boxes_arr[order[1:], 3])
        
        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        inter = w * h
        
        areas = (boxes_arr[order[1:], 2] - boxes_arr[order[1:], 0]) * \
                (boxes_arr[order[1:], 3] - boxes_arr[order[1:], 1])
        area_i = (boxes_arr[i, 2] - boxes_arr[i, 0]) * (boxes_arr[i, 3] - boxes_arr[i, 1])
        
        iou = inter / (area_i + areas - inter + 1e-6)
        inds = np.where(iou <= 0.5)[0]
        order = order[inds + 1]
    
    results = []
    for idx in keep:
        xyxy = boxes_arr[idx]
        results.append({
            'bbox': [float(xyxy[0]), float(xyxy[1]), 
                    float(xyxy[2]-xyxy[0]), float(xyxy[3]-xyxy[1])],
            'score': float(scores_arr[idx])
        })
    return results

predictions_multiscale = []

for img_data in tqdm(gt['images'], desc="Multi-scale Detection"):
    img_path = os.path.join(IMAGE_FOLDER, img_data['file_name'])
    if not os.path.exists(img_path):
        continue
    
    detections = detect_multiscale(model, img_path)
    for det in detections:
        predictions_multiscale.append({
            "image_id": img_data['id'],
            "category_id": 1,
            "bbox": det['bbox'],
            "score": det['score']
        })

print(f"\nMulti-scale predictions: {len(predictions_multiscale)}")

if predictions_multiscale:
    coco_dt_ms = coco_gt.loadRes(predictions_multiscale)
    coco_eval_ms = COCOeval(coco_gt, coco_dt_ms, 'bbox')
    coco_eval_ms.params.catIds = [1]
    coco_eval_ms.evaluate()
    coco_eval_ms.accumulate()
    coco_eval_ms.summarize()

## Method 4: Ensemble (SAHI + Multi-scale)

Combine the best methods for maximum recall.

In [ ]:
print("="*60)
print("METHOD 4: Ensemble (SAHI + Multi-scale)")
print("="*60)

# Combine predictions
combined_by_image = defaultdict(list)

for pred in predictions_sahi:
    combined_by_image[pred['image_id']].append(pred)

for pred in predictions_multiscale:
    combined_by_image[pred['image_id']].append(pred)

# Apply NMS per image
predictions_ensemble = []

for img_id, preds in combined_by_image.items():
    if not preds:
        continue
    
    boxes = np.array([[p['bbox'][0], p['bbox'][1],
                      p['bbox'][0]+p['bbox'][2], p['bbox'][1]+p['bbox'][3]]
                     for p in preds])
    scores = np.array([p['score'] for p in preds])
    
    # NMS
    keep = []
    order = scores.argsort()[::-1]
    
    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break
        
        xx1 = np.maximum(boxes[i, 0], boxes[order[1:], 0])
        yy1 = np.maximum(boxes[i, 1], boxes[order[1:], 1])
        xx2 = np.minimum(boxes[i, 2], boxes[order[1:], 2])
        yy2 = np.minimum(boxes[i, 3], boxes[order[1:], 3])
        
        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        inter = w * h
        
        areas = (boxes[order[1:], 2] - boxes[order[1:], 0]) * \
                (boxes[order[1:], 3] - boxes[order[1:], 1])
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
        
        iou = inter / (area_i + areas - inter + 1e-6)
        inds = np.where(iou <= 0.5)[0]
        order = order[inds + 1]
    
    for idx in keep:
        predictions_ensemble.append(preds[idx])

print(f"\nEnsemble predictions: {len(predictions_ensemble)}")

if predictions_ensemble:
    coco_dt_ens = coco_gt.loadRes(predictions_ensemble)
    coco_eval_ens = COCOeval(coco_gt, coco_dt_ens, 'bbox')
    coco_eval_ens.params.catIds = [1]
    coco_eval_ens.evaluate()
    coco_eval_ens.accumulate()
    coco_eval_ens.summarize()

## Results Comparison

In [ ]:
print("\n" + "="*70)
print("RESULTS COMPARISON")
print("="*70)
print(f"\n{'Method':<25} {'Predictions':>12} {'vs GT (169)':>15}")
print("-"*55)
print(f"{'Your Original':<25} {35:>12} {35/169*100:>14.1f}%")
print(f"{'Standard YOLO':<25} {len(predictions_baseline):>12} {len(predictions_baseline)/169*100:>14.1f}%")
print(f"{'Optimized SAHI':<25} {len(predictions_sahi):>12} {len(predictions_sahi)/169*100:>14.1f}%")
print(f"{'Multi-scale':<25} {len(predictions_multiscale):>12} {len(predictions_multiscale)/169*100:>14.1f}%")
print(f"{'Ensemble':<25} {len(predictions_ensemble):>12} {len(predictions_ensemble)/169*100:>14.1f}%")
print("-"*55)
print(f"\nGround Truth boats: 169")

## Visualize Results

In [ ]:
# Create visualization folder
vis_folder = os.path.join(OUTPUT_FOLDER, "visualizations")
os.makedirs(vis_folder, exist_ok=True)

# Group predictions by image
pred_by_image = defaultdict(list)
for pred in predictions_ensemble:  # Use ensemble (best) results
    pred_by_image[pred['image_id']].append(pred)

# Group GT by image
gt_by_image = defaultdict(list)
for ann in gt['annotations']:
    if ann['category_id'] == 1:
        gt_by_image[ann['image_id']].append(ann)

id_to_filename = {img['id']: img['file_name'] for img in gt['images']}

# Visualize first 10 images with GT boats
for img_id in list(gt_by_image.keys())[:10]:
    img_path = os.path.join(IMAGE_FOLDER, id_to_filename[img_id])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Draw GT in GREEN
    for ann in gt_by_image[img_id]:
        x, y, w, h = [int(v) for v in ann['bbox']]
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(img, 'GT', (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Draw predictions in RED
    for pred in pred_by_image[img_id]:
        x, y, w, h = [int(v) for v in pred['bbox']]
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(img, f"{pred['score']:.2f}", (x, y-5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.title(f"{id_to_filename[img_id]}\nGT: {len(gt_by_image[img_id])} (green), Pred: {len(pred_by_image[img_id])} (red)")
    plt.axis('off')
    plt.savefig(os.path.join(vis_folder, id_to_filename[img_id]))
    plt.show()

print(f"\nVisualizations saved to: {vis_folder}")

## Save Best Predictions

In [ ]:
# Save the best predictions (ensemble)
pred_json_path = os.path.join(OUTPUT_FOLDER, "predictions_improved.json")
with open(pred_json_path, 'w') as f:
    json.dump(predictions_ensemble, f, indent=2)

print(f"Predictions saved to: {pred_json_path}")
print(f"Total predictions: {len(predictions_ensemble)}")

## 🎯 Summary: What Changed

| Issue | Original | Fixed |
|-------|----------|-------|
| **SAHI slicing** | 640×640 (1 slice = nothing!) | 256×256 (~9 slices) |
| **Overlap** | 20% | 30% |
| **Confidence** | 0.3 | 0.15 |
| **Methods** | Single | Ensemble (SAHI + Multi-scale) |

### Why This Works:

1. **Smaller slices** = boats appear larger in each slice = easier to detect
2. **More overlap** = boats at slice boundaries get detected
3. **Lower confidence** = distant/small boats not filtered out
4. **Ensemble** = catches boats that one method might miss